In [ ]:
# !pip install keras-tuner

In [ ]:
import numpy as np
import sys
sys.path.append('..')
import pickle
import logging
import os.path as osp
import tensorflow as tf
from moisture_rnn_pkl import pkl2train
from moisture_rnn import RNNParams, RNNData, RNN, rnn_data_wrap
from utils import hash2, read_yml, read_pkl, retrieve_url, Dict, print_dict_summary, print_first, str2time, logging_setup
from moisture_rnn import RNN
import reproducibility
from data_funcs import rmse, to_json, combine_nested, subset_by_features, read_and_clean
from moisture_models import run_augmented_kf
from metrics import ros_3wind
import copy
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import time
from sklearn.model_selection import ParameterGrid

## Param Setup

In [ ]:
params = read_yml("../params.yaml", subkey="rnn")
params

In [ ]:
param_grids = {
    'batch_size': [16, 32, 64],
    'timesteps': [6, 12, 24],
    'learning_rate': [0.0001, 0.001, 0.01, 0.1],
    'batch_schedule_type': [None],
    'stateful': [False],
    'early_stopping_patience': [5]    
}

In [ ]:
print(len(list(ParameterGrid(param_grids))))
list(ParameterGrid(param_grids))[0:5]

In [ ]:
models_grid = {
    'lstm_only':{
        'hidden_layers': ['dense', 'lstm', 'dense', 'dense'],
        'hidden_units': [64, 32, 32, 16],
        'hidden_activation': ['relu','tanh', 'relu', 'relu']
    },
    'lstm_and_1dconv':{
        'hidden_layers': ['dense', 'lstm', 'conv1d', 'dense'],
        'hidden_units': [64, 32, 32, 16],
        'hidden_activation': ['relu','tanh', 'relu', 'relu']        
    },
    'lstm_and_attention':{
        'hidden_layers': ['dense', 'lstm', 'attention', 'dense'],
        'hidden_units': [64, 32, None, 16],
        'hidden_activation': ['relu','tanh', 'relu', 'relu']        
    }
}

In [ ]:
models_grid

## Read Data

In [ ]:
# Params used for data filtering
params_data = read_yml("../params_data.yaml") 
params_data

In [ ]:
filename = "fmda_rocky_202308-202408_f03.pkl"
retrieve_url(
    url = f"https://demo.openwfm.org/web/data/fmda/dicts/{filename}", 
    dest_path = f"../data/{filename}")

file_paths = [f'../data/{filename}']

In [ ]:
train = read_and_clean(file_paths, atm_source="HRRR", params_data = params_data, verbose=True)